# 05. Mediation and Path Analysis (Bootstrapped)Hypothesized chain: $\text{Bank\_Barrier} \rightarrow \text{Motivation} \rightarrow \text{GPA}$ — Baron–Kenny steps, Sobel test, 5,000-replication bootstrap CI for the indirect effect.

In [ ]:
import pandas as pdimport numpy as npimport statsmodels.api as smfrom scipy import statsdf = pd.read_csv('/mnt/data/cleaned_dataset.csv')X = sm.add_constant(df['Bank_Barrier'])m_c  = sm.OLS(df['GPA'], X).fit()                                   # c: total effectm_a  = sm.OLS(df['Motivation'], X).fit()                            # a: IV -> Mm_bc = sm.OLS(df['GPA'], sm.add_constant(df[['Bank_Barrier','Motivation']])).fit()  # b & c'a, se_a = m_a.params['Bank_Barrier'], m_a.bse['Bank_Barrier']b, se_b = m_bc.params['Motivation'], m_bc.bse['Motivation']c, c_prime = m_c.params['Bank_Barrier'], m_bc.params['Bank_Barrier']indirect = a*bsobel_se = np.sqrt(b**2*se_a**2 + a**2*se_b**2)sobel_z, sobel_p = indirect/sobel_se, 2*(1-stats.norm.cdf(abs(indirect/sobel_se)))print(f"a={a:.4f} (p={m_a.pvalues['Bank_Barrier']:.2e}) | b={b:.4f} (p={m_bc.pvalues['Motivation']:.2e})")print(f"c={c:.4f} | c'={c_prime:.4f} (p={m_bc.pvalues['Bank_Barrier']:.4f})")print(f"Indirect={indirect:.4f} | Sobel Z={sobel_z:.4f}, p={sobel_p:.2e}")print(f"Proportion mediated: {indirect/c:.2%}")

In [ ]:
# 5,000-replication non-parametric bootstrapnp.random.seed(42)n = len(df)boots = []for _ in range(5000):    s = df.sample(n=n, replace=True)    ma = sm.OLS(s['Motivation'], sm.add_constant(s['Bank_Barrier'])).fit()    mb = sm.OLS(s['GPA'], sm.add_constant(s[['Bank_Barrier','Motivation']])).fit()    boots.append(ma.params['Bank_Barrier'] * mb.params['Motivation'])lo, hi = np.percentile(boots, [2.5, 97.5])print(f"95% Bootstrap CI: [{lo:.4f}, {hi:.4f}]")print("CI excludes zero -> significant indirect effect." if (lo>0 or hi<0) else "CI includes zero.")import matplotlib.pyplot as pltplt.hist(boots, bins=40)plt.axvline(lo, color='r', ls='--'); plt.axvline(hi, color='r', ls='--')plt.title('Bootstrap Distribution of Indirect Effect (a×b)')plt.savefig('/mnt/data/Fig_Mediation_Bootstrap.png', dpi=300)plt.show()